<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/Mercosul/Parlamento%20uruguaio/parlamento_uy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Coleta notícias no portal oficial do Parlamento do Uruguai. Se ele estiver indisponível, tenta [notícias oficiais da Câmara de Representantes](https://www.diputados.gub.uy/noticias/). Se os dois portais falharem, coleta [eventos da Biblioteca do Poder Legislativo](https://biblioteca.parlamento.gub.uy/eventos/), com fonte e categoria separadas. **Eventos da Biblioteca não são notícias legislativas.**


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [ ]:
# Os portais de notícias podem responder HTTP 403/502 em algumas redes.
BASE_URL = "https://parlamento.gub.uy/noticiasyeventos/noticias?field_noticia_fecha_value%5Bmin%5D=2000-07-26&field_noticia_fecha_value%5Bmax%5D=&field_noticia_cuerpo_value=All&body_value="
CHAMBER_NEWS_URL = "https://www.diputados.gub.uy/noticias/"
ALTERNATE_URL = "https://biblioteca.parlamento.gub.uy/eventos/"
DATABASE_NAME = "parlamento_uy.db"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; FAPESP-Research-Scraper/1.0)"}

print("Fontes: Parlamento UY → Câmara de Representantes → eventos da Biblioteca (cobertura limitada).")


In [ ]:
def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            category TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela prontos")

create_database()

In [ ]:
def insert_article(title, date, category, url, source="Parlamento UY"):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, category, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, category, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [ ]:
def get_total_pages():
    try:
        print("Detectando número de páginas...")
        r = requests.get(BASE_URL, headers=HEADERS, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        # Tentativa 1: Busca pelo título 'Ir a la última página'
        # Exemplo HTML: <a href="..." title="Ir a la última página" ...>
        last_page_link = soup.find("a", title="Ir a la última página")

        # Tentativa 2: Busca por texto '»'
        if not last_page_link:
            last_page_link = soup.find("a", string=re.compile(r'»'))

        # Tentativa 3: Busca paginação genérica e pega o maior número
        if not last_page_link:
             print("Paginação 'última' não encontrada, buscando maior número nos links...")
             pagination_links = soup.select("ul.pagination li.page-item a.page-link")
             max_page = 0
             for link in pagination_links:
                 href = link.get('href', '')
                 match = re.search(r'page=(\d+)', href)
                 if match:
                     page_num = int(match.group(1))
                     if page_num > max_page:
                         max_page = page_num
             if max_page > 0:
                 return max_page

        if last_page_link:
            href = last_page_link['href']
            print(f"Link última encontrado: {href}")
            match = re.search(r'page=(\d+)', href)
            if match:
                return int(match.group(1))

        # Fallback manual em caso de falha na detecção automática
        # O usuário informou que espera cerca de 1113 páginas
        raise RuntimeError("Paginação do Parlamento UY não identificada; confira o endereço e os seletores antes da coleta.")

    except Exception as e:
        print(f"Erro ao determinar total de páginas: {e}")
        raise

MAX_PAGES = 1  # páginas recentes; para histórico, use get_total_pages() + 1
total_pages = MAX_PAGES - 1
print(f"Páginas selecionadas: {MAX_PAGES}")

In [ ]:
from urllib.parse import urljoin, urlparse

inserted_count = 0
seen_count = 0
fallback_used = False

for page in range(total_pages + 1):
    url = f"{BASE_URL}&page={page}"
    print(f"Coletando notícias parlamentares: página {page + 1}/{MAX_PAGES}...")
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        fallback_used = True
        print(f"Portal do Parlamento indisponível ({type(exc).__name__}: {exc}).")
        break

    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.select("article")
    if not articles:
        fallback_used = True
        print("Portal do Parlamento sem artigos reconhecíveis; tentando a Câmara.")
        break
    for article in articles:
        link_tag = article.select_one("h2.node__title a[href]")
        if not link_tag:
            continue
        title = link_tag.get_text(" ", strip=True)
        link = urljoin("https://parlamento.gub.uy", link_tag["href"])
        if urlparse(link).hostname != "parlamento.gub.uy":
            continue
        date_tag = article.select_one("time")
        date = (date_tag.get("datetime") or date_tag.get_text(" ", strip=True)) if date_tag else ""
        cat_tag = article.select_one(".field--name-field-noticia-cuerpo")
        category = cat_tag.get_text(" ", strip=True) if cat_tag else "Parlamento"
        if title:
            seen_count += 1
            if insert_article(title, date, category, link, "Parlamento UY — notícias"):
                inserted_count += 1
    if page < total_pages:
        time.sleep(0.5)

if seen_count == 0:
    fallback_used = True

if fallback_used:
    print(f"Tentando notícias oficiais da Câmara de Representantes: {CHAMBER_NEWS_URL}")
    chamber_count = 0
    try:
        response = requests.get(CHAMBER_NEWS_URL, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        print(f"Câmara indisponível ({type(exc).__name__}: {exc}).")
    else:
        soup = BeautifulSoup(response.text, "html.parser")
        candidate_links = soup.select('article a[href*="/noticias/"], .post a[href*="/noticias/"], main a[href*="/noticias/"]')
        urls_seen = set()
        for link_tag in candidate_links:
            link = urljoin(response.url, link_tag.get("href", ""))
            parsed = urlparse(link)
            if parsed.hostname not in {"www.diputados.gub.uy", "diputados.gub.uy"} or not parsed.path.startswith("/noticias/"):
                continue
            if parsed.path.rstrip("/") == "/noticias" or link in urls_seen:
                continue
            title = link_tag.get_text(" ", strip=True)
            if len(title) < 12:
                continue
            container = link_tag.find_parent(["article", "div", "li"])
            date_tag = container.select_one("time, .entry-date, .post-date, .date") if container else None
            date = (date_tag.get("datetime") or date_tag.get_text(" ", strip=True)) if date_tag else ""
            urls_seen.add(link)
            chamber_count += 1
            seen_count += 1
            if insert_article(title, date, "Câmara de Representantes", link,
                              "Câmara de Representantes UY — notícias"):
                inserted_count += 1
        if not chamber_count:
            print("Página da Câmara abriu, mas não produziu notícias válidas.")

    if chamber_count == 0:
        response = requests.get(ALTERNATE_URL, headers=HEADERS, timeout=45)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        cards = soup.select(".evento-container")
        if not cards:
            raise RuntimeError("A Biblioteca não trouxe eventos; confira o seletor.")
        library_count = 0
        for card in cards:
            link_tag = card.select_one('a[href*="/eventos/ver/"]')
            date_tag = card.select_one("h3")
            if not link_tag or not date_tag:
                continue
            link = urljoin(response.url, link_tag.get("href", ""))
            if urlparse(link).hostname != "biblioteca.parlamento.gub.uy":
                continue
            title = link_tag.get_text(" ", strip=True)
            date = date_tag.get_text(" ", strip=True)
            if title and date:
                seen_count += 1
                library_count += 1
                if insert_article(title, date, "Eventos da Biblioteca", link,
                                  "Biblioteca do Poder Legislativo — eventos"):
                    inserted_count += 1
        if not library_count:
            raise RuntimeError("A Biblioteca abriu, mas não produziu registros válidos.")
        print("Cobertura limitada a eventos da Biblioteca; notícias legislativas indisponíveis nesta execução.")

if not seen_count:
    raise RuntimeError("Nenhum registro válido coletado do Parlamento, da Câmara ou da Biblioteca.")
print(f"📥 Registros vistos: {seen_count}; novos registros inseridos: {inserted_count}")


In [ ]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("SELECT * FROM articles", conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head())

In [ ]:
import plotly.io as pio
# Garantir que o renderizador seja o do Colab
pio.renderers.default = "colab"

# Carregar dados atualizados
df_db = load_articles()

if not df_db.empty:
    # 1. Distribuição por Categoria
    cat_count = df_db['category'].value_counts().reset_index()
    cat_count.columns = ['Category', 'Count']

    fig1 = px.pie(cat_count, names='Category', values='Count', title='Registros por categoria (notícias ou eventos)')
    fig1.show()

    fig2 = px.bar(cat_count, x='Category', y='Count', title='Registros por categoria', color='Category')
    fig2.show()
else:
    print("Sem dados no banco 'parlamento_uy.db' para visualizar.")